# Torlak Morphological Feature Difficulty Analysis
## XLM-RoBERTa-large · Feature-pair probing

**Goal:** Determine which morphological features are easy vs. hard for the model to predict.

**Approach:** Train one XLM-RoBERTa-large model per individual morphological feature, each with **two heads**:
- **Head 1 (anchor):** UPOS — always present, gives part-of-speech context
- **Head 2 (probe):** one feature type at a time (Gender, Case, Number, Person, Tense, VerbForm, Mood, Degree, …)

Individual features are extracted from the CoNLL-U `FEATS` column produced by the MTE→UD mapping
(e.g. `Case=Nom|Gender=Masc|Number=Sing` → three separate tasks).

**Output:** ranked table + bar chart of per-feature test accuracy.

## 0 · Setup

In [ ]:
# ========================= 
# 0) SETUP
# =========================
!pip -q install lxml

import os, re, json, math, time, random
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional
from collections import Counter, defaultdict

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup

from google.colab import drive
drive.mount("/content/drive")

print("🧠 torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

## 1 · Configuration

In [ ]:
# =========================
# 1) CONFIG
# =========================

DATA_ROOT     = Path("/content/drive/MyDrive/TorlakData")
MTE2UD_PATH   = DATA_ROOT / "mte2ud_output.txt"
SPK_META_PATH = DATA_ROOT / "spk.metadata.txt"
EXB_DIR       = DATA_ROOT / "exb_corrected"
GEO_TSV       = None  # e.g. DATA_ROOT / "place_geo.tsv"

MODELS_ROOT = Path("/content/drive/MyDrive/TorlakTag/feat_difficulty")
MODELS_ROOT.mkdir(parents=True, exist_ok=True)

# XLM-RoBERTa-large — the winner from the multi-run notebook
MODEL_NAME = "FacebookAI/xlm-roberta-large"
EPOCHS        = 7
WARMUP_RATIO  = 0.06
WEIGHT_DECAY  = 0.01
GRAD_CLIP     = 1.0
PATIENCE      = 4
FREEZE_EPOCHS = 1

BS       = 24
ACCUM    = 4      # effective batch = 96
LR       = 1e-5
MAX_LEN  = 16
USE_FP16 = True

# Minimum training examples a feature VALUE must have to be kept in vocab
MIN_FEAT_VAL_FREQ = 5
# Minimum training examples a feature TYPE must have to get its own model run
MIN_FEAT_TYPE_EXAMPLES = 100

SEED = 13
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("📁 MODELS_ROOT:", MODELS_ROOT)
print("🧠 device:", device)

## 2 · Token normalisation

*Copied verbatim from the original multi-run notebook — do not modify.*

In [ ]:
# =========================
# 2) TOKEN NORMALIZATION (from original notebook)
# =========================

def apply_spec_mapping(s: str) -> str:
    if s is None:
        return ""
    s = s.replace("#", "")
    s = s.replace("W", "Ə").replace("w", "ə")
    s = s.replace("1", "ḱ").replace("6", "ḱ")
    s = s.replace("2", "ǵ")
    s = s.replace("3", "č")
    s = s.replace("x", "š").replace("X", "š")
    s = s.replace("5", "ƨ")
    s = s.replace("ššš", "XXX")
    return s

RE_OVERLAP   = re.compile(r"\[[^\]]*\]")
RE_DOUBLEPAR = re.compile(r"^\(\(.*\)\)$")
RE_BULLETS   = re.compile(r"^[•]+$")
RE_LONGVOWEL = re.compile(r"([aeiouə])\1+")
RE_SPACES    = re.compile(r"\s+")

_STRIP_EDGE = " \t\r\n\"'“”„`´.,;:!?(){}[]<>•"

def is_x_special_token(tok: str) -> bool:
    t = tok.strip()
    if not t:
        return True
    if RE_DOUBLEPAR.match(t):
        return True
    if RE_BULLETS.match(t):
        return True
    return False

def strip_attached_specials(tok: str) -> str:
    t = tok.strip()
    t = re.sub(r"/+$", "", t)      # rek/ -> rek
    t = t.strip(_STRIP_EDGE)
    return t

def normalize_word(tok: str) -> str:
    t = apply_spec_mapping(tok).lower()
    t = RE_LONGVOWEL.sub(r"\1", t)
    t = RE_SPACES.sub(" ", t).strip()
    return t

def tokenize_with_rules(raw_text: str):
    if raw_text is None:
        return [], []
    s = apply_spec_mapping(raw_text).lower()
    s = RE_OVERLAP.sub(" ", s)

    raw_tokens = [t for t in s.split() if t.strip()]
    tokens, special = [], []
    for rt in raw_tokens:
        if is_x_special_token(rt):
            tokens.append(rt)
            special.append(True)
            continue

        has_alnum = any(ch.isalpha() or ch.isdigit() for ch in rt)
        if has_alnum:
            w = strip_attached_specials(rt)
            w = normalize_word(w)
            if w:
                tokens.append(w)
                special.append(False)
            else:
                tokens.append(rt)
                special.append(True)
        else:
            tokens.append(rt)
            special.append(True)

    return tokens, special

print(tokenize_with_rules("((?)) stoju/ rek/ •• aaa əəə [overlap] test"))

## 3 · Load MTE→UD mapping + data splits

*Copied verbatim from the original multi-run notebook.*

In [ ]:
# =========================
# 3) LOAD MTE→UD MAPPING + SPLITS
# =========================

UPOS_SET = {
    "ADJ","ADP","ADV","AUX","CCONJ","DET","INTJ","NOUN","NUM","PART",
    "PRON","PROPN","PUNCT","SCONJ","SYM","VERB","X"
}

def load_mte2ud(path: Path):
    m = {}
    bad = 0
    with path.open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            parts = re.split(r"\t+", line)
            if len(parts) < 3:
                parts = re.split(r"\s{2,}", line)
            if len(parts) < 3:
                bad += 1
                continue

            mte  = parts[0].strip()
            upos = parts[1].strip()

            if len(parts) >= 4 and parts[2].strip() in UPOS_SET and upos in UPOS_SET:
                feats = parts[3].strip()
            else:
                feats = parts[2].strip()

            feats = feats if feats else "_"
            m[mte] = (upos if upos else "X", feats)

    print(f"✅ loaded MTE→UD mapping: {len(m)} tags (skipped {bad} broken lines)")
    return m

mte2ud = load_mte2ud(MTE2UD_PATH)

def find_split_file(root: Path, stem: str) -> Path:
    for ext in ["", ".txt", ".tsv", ".conllu", ".conll", ".data"]:
        p = root / f"{stem}{ext}"
        if p.exists():
            return p
    hits = [h for h in root.rglob(f"{stem}*") if h.is_file()]
    if hits:
        hits = sorted(hits, key=lambda x: len(str(x)))
        return hits[0]
    raise FileNotFoundError(f"Could not find split file for '{stem}' under {root}")

TRAIN_PATH = find_split_file(DATA_ROOT, "tor_train")
DEV_PATH   = find_split_file(DATA_ROOT, "tor_dev")
TEST_PATH  = find_split_file(DATA_ROOT, "tor_test")

print("📄 train:", TRAIN_PATH)
print("📄 dev  :", DEV_PATH)
print("📄 test :", TEST_PATH)

def read_tok_lemma_mte(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split("\t")
            if len(parts) < 3:
                parts = line.split()
            if len(parts) < 3:
                continue
            form, lemma, mte = parts[0], parts[1], parts[2]

            # normalize similarly to EXB processing
            tks, sp = tokenize_with_rules(form)
            if not tks:
                continue
            form_n = tks[0]
            is_special = sp[0]

            if is_special:
                rows.append((form_n, form_n, "X"))  # forced X tag
                continue

            lemma_n = normalize_word(strip_attached_specials(lemma))
            if not lemma_n:
                lemma_n = form_n

            rows.append((form_n, lemma_n, mte))
    return rows

train_raw = read_tok_lemma_mte(TRAIN_PATH)
dev_raw   = read_tok_lemma_mte(DEV_PATH)
test_raw  = read_tok_lemma_mte(TEST_PATH)

print(f"✅ loaded splits: train={len(train_raw)} dev={len(dev_raw)} test={len(test_raw)}")
print("sample:", train_raw[:5])

def to_ud_example(row):
    form, lemma, xpos = row
    if xpos == "X":
        return (form, lemma, "X", "_", "X")
    upos, feats = mte2ud.get(xpos, ("X", "_"))
    return (form, lemma, upos, feats, xpos)

train_ex = [to_ud_example(r) for r in train_raw]
dev_ex   = [to_ud_example(r) for r in dev_raw]
test_ex  = [to_ud_example(r) for r in test_raw]

print("Converted sample:", train_ex[:5])

## 4 · Label maps

*Copied verbatim. `upos2id` is the shared UPOS vocab used across all feature runs.*

In [ ]:
# =========================
# 4) LABEL MAPS
# =========================

def build_vocab(items, min_freq=1, specials=None):
    specials = specials or []
    c = Counter(items)
    vocab = {}
    for sp in specials:
        vocab[sp] = len(vocab)
    for k, v in c.most_common():
        if k in vocab:
            continue
        if v >= min_freq:
            vocab[k] = len(vocab)
    return vocab

def ensure_special(v: Dict[str, int], key: str):
    if key not in v:
        v[key] = len(v)
    return v

upos_items = [u for _, _, u, _, _ in train_ex]
upos2id    = build_vocab(upos_items, min_freq=1, specials=["X", "_"])
upos2id    = ensure_special(upos2id, "X")
upos2id    = ensure_special(upos2id, "_")
id2upos    = {i: s for s, i in upos2id.items()}

print(f"✅ UPOS vocab: {len(upos2id)} classes → {list(upos2id.keys())}")

## 5 · Discover individual morphological features

Parse the `FEATS` strings from the training data to find every feature type and its frequency.
Feature types with fewer than `MIN_FEAT_TYPE_EXAMPLES` training tokens are skipped.

In [ ]:
# =========================
# 5) DISCOVER FEATURE TYPES
# =========================

def parse_feats(feats_str: str) -> Dict[str, str]:
    """'Case=Nom|Gender=Masc|Number=Sing' → {'Case': 'Nom', 'Gender': 'Masc', 'Number': 'Sing'}"""
    if not feats_str or feats_str == "_":
        return {}
    result = {}
    for kv in feats_str.split("|"):
        if "=" in kv:
            k, v = kv.split("=", 1)
            result[k.strip()] = v.strip()
    return result

feat_type_counts  = Counter()
feat_value_counts = defaultdict(Counter)

for _, _, _, feats, _ in train_ex:
    for k, v in parse_feats(feats).items():
        feat_type_counts[k] += 1
        feat_value_counts[k][v] += 1

FEAT_TYPES = sorted(
    [ft for ft, cnt in feat_type_counts.items() if cnt >= MIN_FEAT_TYPE_EXAMPLES],
    key=lambda ft: -feat_type_counts[ft]
)

print(f"Found {len(FEAT_TYPES)} feature types with ≥{MIN_FEAT_TYPE_EXAMPLES} training examples:\n")
print(f"{'Feature Type':<18} {'# train ex':>10}  {'# values':>8}  Top values")
print("─" * 80)
for ft in FEAT_TYPES:
    vals = feat_value_counts[ft]
    top  = ", ".join(f"{v}({c})" for v, c in vals.most_common(6))
    print(f"{ft:<18} {feat_type_counts[ft]:>10}  {len(vals):>8}  {top}")

## 6 · Dataset, model, helpers

Two-head architecture using the same `mean_pool` and `get_hidden_size_from_config` from
the original notebook. Only the head structure differs: **UPOS head** + **one feature head**.

Tokens that don't carry the target feature get label `_` (absent class, id=0).
The key evaluation metric is accuracy **only on tokens where the feature is present**,
so the `_` majority class cannot inflate scores.

In [ ]:
# =========================
# 6) DATASET / MODEL
# =========================

# ── Dataset ───────────────────────────────────────────────────────────────────

class FeatPairDataset(Dataset):
    """
    Each example: (form, upos_id, feat_value_id).
    train_ex tuples are (form, lemma, upos, feats, xpos) — same format as original.
    feat_value_id = id for this feature type's value, or feat2id['_'] if absent.
    """
    def __init__(self, examples, feat_type: str, feat2id: Dict[str, int]):
        self.items = []
        for form, _lemma, upos, feats, _xpos in examples:
            up_id = upos2id.get(upos, upos2id["X"])
            fval  = parse_feats(feats).get(feat_type, "_")
            fe_id = feat2id.get(fval, feat2id.get("<UNK>", feat2id["_"]))
            self.items.append((form, up_id, fe_id))

    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def make_collate(tokenizer, max_length: int):
    def collate(items):
        tokens = [it[0] for it in items]
        up_y   = torch.tensor([it[1] for it in items], dtype=torch.long)
        fe_y   = torch.tensor([it[2] for it in items], dtype=torch.long)
        enc = tokenizer(tokens, padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt")
        return enc["input_ids"], enc["attention_mask"], up_y, fe_y
    return collate

# ── Model helpers (verbatim from original) ────────────────────────────────────

def get_hidden_size_from_config(cfg):
    if hasattr(cfg, "hidden_size"):
        return int(cfg.hidden_size)
    if hasattr(cfg, "d_model"):
        return int(cfg.d_model)
    raise ValueError("Cannot infer hidden size from config.")

def mean_pool(last_hidden, attention_mask):
    # last_hidden: [B, T, H], mask: [B, T]
    mask    = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed  = (last_hidden * mask).sum(dim=1)
    denom   = mask.sum(dim=1).clamp(min=1.0)
    return summed / denom

# ── Two-head model ────────────────────────────────────────────────────────────

class TwoHeadTagger(nn.Module):
    """XLM-RoBERTa-large with UPOS head + one feature head."""
    def __init__(self, encoder_name: str, n_upos: int, n_feat: int, dropout: float = 0.1):
        super().__init__()
        cfg = AutoConfig.from_pretrained(encoder_name)
        self.encoder   = AutoModel.from_pretrained(encoder_name, torch_dtype=torch.float32)
        h              = get_hidden_size_from_config(cfg)
        self.drop      = nn.Dropout(dropout)
        self.upos_head = nn.Linear(h, n_upos)
        self.feat_head = nn.Linear(h, n_feat)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)
        x   = self.drop(mean_pool(out.last_hidden_state, attention_mask))
        return self.upos_head(x), self.feat_head(x)

# ── Evaluate ──────────────────────────────────────────────────────────────────

@torch.no_grad()
def evaluate(model, loader, feat2id: Dict[str, int], amp_dtype=None):
    model.eval()
    total = corr_up = corr_fe = corr_fe_present = n_present = 0
    absent_id = feat2id["_"]

    for input_ids, attn, up_y, fe_y in loader:
        input_ids, attn = input_ids.to(device), attn.to(device)
        up_y, fe_y      = up_y.to(device), fe_y.to(device)

        if amp_dtype is not None and device.type == "cuda":
            with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                up_l, fe_l = model(input_ids, attn)
        else:
            up_l, fe_l = model(input_ids, attn)

        up_p = up_l.argmax(dim=1)
        fe_p = fe_l.argmax(dim=1)

        corr_up += int((up_p == up_y).sum().item())
        corr_fe += int((fe_p == fe_y).sum().item())

        present_mask     = fe_y != absent_id
        n_present       += int(present_mask.sum().item())
        corr_fe_present += int(((fe_p == fe_y) & present_mask).sum().item())
        total           += input_ids.size(0)

    return {
        "upos_acc":         corr_up / max(1, total),
        "feat_acc":         corr_fe / max(1, total),           # includes absent tokens
        "feat_acc_present": corr_fe_present / max(1, n_present),  # only present tokens
        "n":         total,
        "n_present": n_present,
    }

print("✅ model/dataset helpers ready")

## 7 · Training loop

Same freeze/unfreeze, cosine schedule, AMP, early-stopping pattern as the original.

In [ ]:
# =========================
# 7) TRAIN ONE FEATURE MODEL
# =========================

def train_feat_model(feat_type: str, feat2id: Dict[str, int], out_dir: Path) -> Dict:
    """
    Train XLM-RoBERTa-large with UPOS + feat_type heads.
    Saves best_model.pt, feat2id.json, upos2id.json, train_log.json, test_results.json.
    Returns test metrics dict.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    tokenizer.save_pretrained(str(out_dir))
    json.dump(feat2id, open(out_dir / "feat2id.json",  "w", encoding="utf-8"), ensure_ascii=False, indent=2)
    json.dump(upos2id, open(out_dir / "upos2id.json",  "w", encoding="utf-8"), ensure_ascii=False, indent=2)

    collate      = make_collate(tokenizer, MAX_LEN)
    train_ds     = FeatPairDataset(train_ex, feat_type, feat2id)
    dev_ds       = FeatPairDataset(dev_ex,   feat_type, feat2id)
    test_ds      = FeatPairDataset(test_ex,  feat_type, feat2id)

    train_loader = DataLoader(train_ds, batch_size=BS,    shuffle=True,  num_workers=2,
                              collate_fn=collate, pin_memory=True)
    dev_loader   = DataLoader(dev_ds,   batch_size=BS*2,  shuffle=False, num_workers=2,
                              collate_fn=collate, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=BS*2,  shuffle=False, num_workers=2,
                              collate_fn=collate, pin_memory=True)

    model = TwoHeadTagger(MODEL_NAME, len(upos2id), len(feat2id), dropout=0.1).to(device)

    # Freeze encoder for first epoch (same as original)
    for p in model.encoder.parameters():
        p.requires_grad = False

    optim = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY
    )
    total_steps  = EPOCHS * math.ceil(len(train_loader) / ACCUM)
    warmup_steps = int(total_steps * WARMUP_RATIO)
    sched = get_cosine_schedule_with_warmup(optim, num_warmup_steps=warmup_steps,
                                            num_training_steps=total_steps)

    amp_dtype = torch.float16 if (USE_FP16 and device.type == "cuda") else None
    scaler    = torch.amp.GradScaler("cuda") if amp_dtype is not None else None

    best_acc, best_epoch, bad = -1.0, -1, 0

    def rebuild_optimizer_after_unfreeze():
        nonlocal optim, sched
        optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        sched = get_cosine_schedule_with_warmup(optim, num_warmup_steps=warmup_steps,
                                                num_training_steps=total_steps)

    log = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        t0 = time.time()
        loss_accum = 0.0
        seen = 0

        if epoch == FREEZE_EPOCHS + 1:
            for p in model.encoder.parameters():
                p.requires_grad = True
            rebuild_optimizer_after_unfreeze()
            print("🔥 Encoder unfrozen")

        optim.zero_grad(set_to_none=True)

        for step, (input_ids, attn, up_y, fe_y) in enumerate(train_loader, start=1):
            input_ids = input_ids.to(device, non_blocking=True)
            attn      = attn.to(device, non_blocking=True)
            up_y      = up_y.to(device, non_blocking=True)
            fe_y      = fe_y.to(device, non_blocking=True)

            if amp_dtype is not None:
                with torch.amp.autocast(device_type="cuda", dtype=amp_dtype):
                    up_l, fe_l = model(input_ids, attn)
                    loss = (
                        nn.functional.cross_entropy(up_l, up_y) +
                        nn.functional.cross_entropy(fe_l, fe_y)
                    ) / ACCUM
                scaler.scale(loss).backward()
            else:
                up_l, fe_l = model(input_ids, attn)
                loss = (
                    nn.functional.cross_entropy(up_l, up_y) +
                    nn.functional.cross_entropy(fe_l, fe_y)
                ) / ACCUM
                loss.backward()

            loss_accum += float(loss.item()) * input_ids.size(0) * ACCUM
            seen       += input_ids.size(0)

            if (step % ACCUM) == 0:
                if scaler is not None:
                    scaler.unscale_(optim)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    scaler.step(optim)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    optim.step()
                optim.zero_grad(set_to_none=True)
                sched.step()

        # save last checkpoint each epoch (overwrites)
        torch.save(model.state_dict(), out_dir / "last_epoch.pt")

        dev_m      = evaluate(model, dev_loader, feat2id, amp_dtype=amp_dtype)
        train_loss = loss_accum / max(1, seen)
        dt         = time.time() - t0

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "dev_upos_acc":         dev_m["upos_acc"],
            "dev_feat_acc":         dev_m["feat_acc"],
            "dev_feat_acc_present": dev_m["feat_acc_present"],
            "minutes": dt / 60.0,
        }
        log.append(row)
        json.dump(log, open(out_dir / "train_log.json", "w", encoding="utf-8"), indent=2)

        print(f"[{epoch:02d}] train_loss={train_loss:.4f}  "
              f"dev_upos={dev_m['upos_acc']*100:.2f}%  "
              f"dev_feat(present)={dev_m['feat_acc_present']*100:.2f}%  "
              f"n_present={dev_m['n_present']}  ({dt/60:.1f}m)")

        # best model tracked on present-only feature accuracy
        if dev_m["feat_acc_present"] > best_acc + 1e-6:
            best_acc, best_epoch, bad = dev_m["feat_acc_present"], epoch, 0
            torch.save(model.state_dict(), out_dir / "best_model.pt")
            json.dump({"best_dev_feat_acc_present": best_acc, "best_epoch": best_epoch,
                       "feat_type": feat_type, "model_name": MODEL_NAME},
                      open(out_dir / "best_meta.json", "w", encoding="utf-8"), indent=2)
            print("💾 saved best_model.pt")
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"⏹️  Early stopping. best_dev_feat(present)={best_acc*100:.2f}% at epoch {best_epoch}")
                break

    # ── TEST best ─────────────────────────────────────────────────────────────
    best_model = TwoHeadTagger(MODEL_NAME, len(upos2id), len(feat2id), dropout=0.1).to(device)
    best_model.load_state_dict(torch.load(out_dir / "best_model.pt", map_location=device))
    test_m = evaluate(best_model, test_loader, feat2id, amp_dtype=amp_dtype)

    print(f"🧪 TEST  upos={test_m['upos_acc']*100:.2f}%  "
          f"feat(all)={test_m['feat_acc']*100:.2f}%  "
          f"feat(present)={test_m['feat_acc_present']*100:.2f}%  "
          f"n_present={test_m['n_present']}")

    test_m["feat_type"]   = feat_type
    test_m["best_epoch"]  = best_epoch
    test_m["n_values"]    = len(feat2id) - 2   # exclude '_' and '<UNK>'
    test_m["train_count"] = feat_type_counts.get(feat_type, 0)
    json.dump({k: float(v) if isinstance(v, (float, int)) else v for k, v in test_m.items()},
              open(out_dir / "test_results.json", "w"), indent=2)

    torch.cuda.empty_cache()
    return test_m

print("✅ training loop ready")

## 8 · Run all feature models

Each feature is its own cell — if the runtime disconnects, completed features are
already saved to Drive and will be skipped on reconnect.

**7 epochs each.** Time per epoch is printed. On reconnect: run setup cells 0–7,
then run whichever feature cell you left off at.

In [ ]:
# ── Shared: load existing results from Drive ─────────────────────────────────
results_json = MODELS_ROOT / "all_results.json"
all_results  = {}
if results_json.exists():
    all_results = json.load(open(results_json, encoding="utf-8"))
    print(f"Loaded {len(all_results)} completed: {list(all_results.keys())}")
else:
    print("No all_results.json yet — starting fresh.")


In [ ]:
# ── Feature: Case ──────────────────────────────────────────────────────────
_ft = "Case"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "case")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: Gender ──────────────────────────────────────────────────────────
_ft = "Gender"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "gender")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: Person ──────────────────────────────────────────────────────────
_ft = "Person"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "person")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: VerbForm ──────────────────────────────────────────────────────────
_ft = "VerbForm"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "verbform")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: Tense ──────────────────────────────────────────────────────────
_ft = "Tense"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "tense")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: Mood ──────────────────────────────────────────────────────────
_ft = "Mood"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "mood")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: PronType ──────────────────────────────────────────────────────────
_ft = "PronType"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "prontype")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: Degree ──────────────────────────────────────────────────────────
_ft = "Degree"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "degree")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: Voice ──────────────────────────────────────────────────────────
_ft = "Voice"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "voice")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: Reflex ──────────────────────────────────────────────────────────
_ft = "Reflex"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "reflex")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: Animacy ──────────────────────────────────────────────────────────
_ft = "Animacy"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "animacy")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: NumType ──────────────────────────────────────────────────────────
_ft = "NumType"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "numtype")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: Polarity ──────────────────────────────────────────────────────────
_ft = "Polarity"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "polarity")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: Definite ──────────────────────────────────────────────────────────
_ft = "Definite"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "definite")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


In [ ]:
# ── Feature: Poss ──────────────────────────────────────────────────────────
_ft = "Poss"
if _ft in all_results:
    print(f"SKIP {_ft} — already in all_results.json")
else:
    if _ft not in feat_type_counts:
        print(f"SKIP {_ft} — not found in data")
    else:
        print(f"\n{'='*60}")
        print(f"Feature: {_ft}  ({feat_type_counts[_ft]} train examples, {len(feat_value_counts[_ft])} values)")
        print(f"{'='*60}")

        feat2id = {"_": 0}
        for val, cnt in feat_value_counts[_ft].most_common():
            if val != "_" and cnt >= MIN_FEAT_VAL_FREQ:
                feat2id[val] = len(feat2id)
        feat2id["<UNK>"] = len(feat2id)
        print(f"  Classes ({len(feat2id)}): {list(feat2id.keys())}")

        result = train_feat_model(_ft, feat2id, MODELS_ROOT / "poss")
        all_results[_ft] = result

        serialisable = {
            k: {kk: float(vv) if isinstance(vv, (float, int)) else vv for kk, vv in v.items()}
            for k, v in all_results.items()
        }
        json.dump(serialisable, open(results_json, "w", encoding="utf-8"), indent=2)
        print(f"  ✅ {_ft} done & saved → all_results.json")


## 9 · Analysis & visualisation

In [ ]:
# =========================
# 9) RESULTS TABLE
# =========================

rows = []
for ft, m in all_results.items():
    rows.append({
        "Feature":              ft,
        "# Values":             m.get("n_values", "?"),
        "# Train ex":           m.get("train_count", "?"),
        "UPOS acc (%)":         round(m["upos_acc"] * 100, 2),
        "Feat acc all (%)":     round(m["feat_acc"] * 100, 2),
        "Feat acc present (%)": round(m["feat_acc_present"] * 100, 2),
        "Best epoch":           m.get("best_epoch", "?"),
    })

df = (pd.DataFrame(rows)
        .sort_values("Feat acc present (%)", ascending=False)
        .reset_index(drop=True))
df.index += 1   # rank from 1
print(df.to_string())

df.to_csv(MODELS_ROOT / "feature_difficulty_ranking.csv", index_label="Rank")
print("\nSaved →", MODELS_ROOT / "feature_difficulty_ranking.csv")

In [ ]:
# Bar chart: hardest → easiest
df_plot = df.sort_values("Feat acc present (%)", ascending=True)
colors  = cm.RdYlGn(np.linspace(0.15, 0.85, len(df_plot)))  # red=hard, green=easy

fig, ax = plt.subplots(figsize=(10, max(4, len(df_plot) * 0.6)))
bars = ax.barh(df_plot["Feature"], df_plot["Feat acc present (%)"], color=colors)

for bar, val in zip(bars, df_plot["Feat acc present (%)"]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=9)

# Overlay UPOS accuracy as reference tick mark
for i, (_, row) in enumerate(df_plot.iterrows()):
    ax.plot(row["UPOS acc (%)"], i, "k|", markersize=12, markeredgewidth=2,
            label="UPOS acc (same run)" if i == 0 else "")

ax.set_xlabel("Test accuracy on tokens where feature is present (%)", fontsize=11)
ax.set_title("Morphological Feature Difficulty\n"
             "(XLM-RoBERTa-large, UPOS+Feature co-training)", fontsize=12)
ax.set_xlim(0, 105)
ax.axvline(x=50, color="gray", linestyle="--", alpha=0.4, label="50% chance baseline")
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(str(MODELS_ROOT / "feature_difficulty.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Saved →", MODELS_ROOT / "feature_difficulty.png")

In [ ]:
# Heatmap: UPOS acc vs Feature acc side by side
fig, ax = plt.subplots(figsize=(7, max(4, len(df) * 0.55)))
data = df[["UPOS acc (%)", "Feat acc present (%)"]].values
im   = ax.imshow(data, aspect="auto", cmap="RdYlGn", vmin=0, vmax=100)
ax.set_xticks([0, 1])
ax.set_xticklabels(["UPOS acc", "Feature acc\n(present tokens)"], fontsize=10)
ax.set_yticks(range(len(df)))
ax.set_yticklabels(df["Feature"].tolist(), fontsize=10)
plt.colorbar(im, ax=ax, label="Accuracy (%)")
for i in range(len(df)):
    for j in range(2):
        val = data[i, j]
        colour = "black" if 25 < val < 80 else "white"
        ax.text(j, i, f"{val:.1f}", ha="center", va="center", fontsize=9, color=colour)
ax.set_title("UPOS vs Feature Accuracy Heatmap\n(ranked easiest → hardest)", fontsize=11)
plt.tight_layout()
fig.savefig(str(MODELS_ROOT / "feature_heatmap.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Scatter 1: training frequency vs accuracy
fig, ax = plt.subplots(figsize=(8, 5))
x  = df["# Train ex"].astype(float)
y  = df["Feat acc present (%)"].astype(float)
sc = ax.scatter(x, y, c=y, cmap="RdYlGn", vmin=0, vmax=100, s=120, zorder=3)
for _, row in df.iterrows():
    ax.annotate(row["Feature"],
                (row["# Train ex"], row["Feat acc present (%)"]),
                textcoords="offset points", xytext=(5, 3), fontsize=8)
if len(x) > 2:
    z  = np.polyfit(np.log1p(x), y, 1)
    xf = np.linspace(x.min(), x.max(), 200)
    ax.plot(xf, np.poly1d(z)(np.log1p(xf)), "k--", alpha=0.4, label="trend (log x)")
plt.colorbar(sc, ax=ax, label="Feature acc present (%)")
ax.set_xlabel("# Training examples with this feature", fontsize=11)
ax.set_ylabel("Test accuracy on present tokens (%)", fontsize=11)
ax.set_title("Training frequency vs. Feature accuracy", fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(str(MODELS_ROOT / "freq_vs_accuracy.png"), dpi=150, bbox_inches="tight")
plt.show()

# Scatter 2: number of classes vs accuracy
fig, ax = plt.subplots(figsize=(7, 5))
x2  = df["# Values"].astype(float)
sc2 = ax.scatter(x2, y, c=y, cmap="RdYlGn", vmin=0, vmax=100, s=120, zorder=3)
for _, row in df.iterrows():
    ax.annotate(row["Feature"],
                (row["# Values"], row["Feat acc present (%)"]),
                textcoords="offset points", xytext=(5, 3), fontsize=8)
plt.colorbar(sc2, ax=ax, label="Feature acc present (%)")
ax.set_xlabel("Number of possible values (classes)", fontsize=11)
ax.set_ylabel("Test accuracy on present tokens (%)", fontsize=11)
ax.set_title("Class count vs. Feature accuracy", fontsize=12)
plt.tight_layout()
fig.savefig(str(MODELS_ROOT / "nclasses_vs_accuracy.png"), dpi=150, bbox_inches="tight")
plt.show()

## 10 · Interpretation guide

| Metric | What it tells you |
|---|---|
| **Feat acc present (%)** | Main difficulty metric — accuracy only on tokens that carry the feature, so the `_` absent-class majority can't inflate the score |
| **UPOS acc (%)** | Should be ~stable across all runs (same anchor head); a drop in a specific run suggests that feature type interferes with POS prediction |
| **Freq vs accuracy** scatter | If low-frequency features are harder → bottleneck is data size; if not → it's linguistic ambiguity or morphological complexity |
| **Class count vs accuracy** scatter | If more-valued features are harder → class confusion / imbalance is the limiting factor |

**Easy features** (high accuracy): well-predicted from the surface form or UPOS alone.  
**Hard features** (low accuracy): candidates for more data, richer context window, or dialect-specific subword models.